# Floorplan SegFormer Training

SegFormer-B2 fine-tuning for floorplan segmentation (10 classes).

**Setup:** Runtime → Change runtime type → A100 GPU

In [ ]:
# 1. Upload training code zip
VERSION = "v3"

from google.colab import files
uploaded = files.upload()  # upload training-code-{VERSION}.zip

In [ ]:
# 2. Unzip and install dependencies
!unzip -o training-code-{VERSION}.zip
!pip install -q torch torchvision transformers opencv-python-headless Pillow numpy albumentations tqdm huggingface-hub svgpathtools

In [ ]:
# 3. Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 4. Generate synthetic data + prepare CubiCasa5K

# Synthetic data
!python generate_synthetic.py --count 2000 --output-dir data/prepared/synthetic/train --seed 42
!python generate_synthetic.py --count 500 --output-dir data/prepared/synthetic/val --seed 999

# CubiCasa5K (~5000 real floorplans) - dataset hosted on Zenodo
!wget -q --show-progress -O cubicasa5k.zip https://zenodo.org/record/2613548/files/cubicasa5k.zip
!unzip -q cubicasa5k.zip -d data/

# Find the actual cubicasa root (with colorful/ inside)
import glob, os
cubicasa_root = "data"
for d in glob.glob("data/**/colorful", recursive=True):
    cubicasa_root = os.path.dirname(d)
    break
print(f"CubiCasa5K root: {cubicasa_root}")

!python prepare_cubicasa.py --data-dir {cubicasa_root} --output-dir data/prepared/cubicasa --val-split 0.1

In [ ]:
# 5. Train SegFormer-B2 on Synthetic + CubiCasa5K
CKPT = f"checkpoints/segformer-floorplan-{VERSION}"
!python train.py \
  --data-dirs data/prepared/cubicasa/train data/prepared/synthetic/train \
  --val-dirs data/prepared/cubicasa/val data/prepared/synthetic/val \
  --output-dir {CKPT} \
  --batch-size 8 \
  --epochs 50

In [ ]:
# 6. Test on a sample image
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

# Pick a random validation image
val_images = os.listdir('data/prepared/synthetic/val/images')
test_img = f'data/prepared/synthetic/val/images/{val_images[0]}'
test_mask = f'data/prepared/synthetic/val/masks/{val_images[0]}'

# Run prediction
!python predict.py --model {CKPT}/best --image {test_img} --output /tmp/test_result.json --save-mask

# Show results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(test_img))
axes[0].set_title('Input')
axes[1].imshow(Image.open(test_mask), cmap='tab10', vmin=0, vmax=9)
axes[1].set_title('Ground Truth')
axes[2].imshow(Image.open('/tmp/test_result_mask.png'), cmap='gray')
axes[2].set_title('Prediction')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 7. Download trained model
from google.colab import files
!zip -r /tmp/segformer-floorplan-{VERSION}-best.zip {CKPT}/best/
files.download(f'/tmp/segformer-floorplan-{VERSION}-best.zip')